## Hybrid Retriever- Combining Dense And Sparse Retriever

In [2]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_core.documents import Document
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableMap
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")


In [3]:
# Step 1: Sample documents
docs = [
    Document(page_content="LangChain helps build LLM applications."),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="LangChain can be used to develop agentic AI applications."),
    Document(page_content="LangChain has many types of retrievers."),
]

# Step 2: Dense Retriever (FAISS + multilingual Hugging Face embeddings)
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
dense_vectorstore = FAISS.from_documents(docs, embedding_model)
dense_retriever = dense_vectorstore.as_retriever(search_kwargs={"k": 3})


In [4]:
### Sparse Retriever (BM25)
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k = 3

## Step 4: Combine with Ensemble Retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3],
)


In [5]:
hybrid_retriever

EnsembleRetriever(retrievers=[VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x14933fcd0>, search_kwargs={'k': 3}), BM25Retriever(vectorizer=<rank_bm25.BM25Okapi object at 0x17ff0bb10>, k=3)], weights=[0.7, 0.3])

In [6]:
# Step 5: Query and get results
query = "How can I build an application using LLMs?"
results = hybrid_retriever.invoke(query)

# Step 6: Print results
for i, doc in enumerate(results):
    print(f"\n🔹 Document {i+1}:\n{doc.page_content}")


🔹 Document 1:
LangChain helps build LLM applications.

🔹 Document 2:
LangChain can be used to develop agentic AI applications.

🔹 Document 3:
Pinecone is a vector database for semantic search.

🔹 Document 4:
LangChain has many types of retrievers.


### RAG Pipeline with hybrid retriever

In [7]:
# Step 5: Prompt Template
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {question}
""")

## Step 6: LLM
llm = init_chat_model("groq:llama-3.1-8b-instant", temperature=0.2)
llm


ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x14d97d610>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x14dac7250>, model_name='llama-3.1-8b-instant', temperature=0.2, model_kwargs={}, groq_api_key=SecretStr('**********'))

In [8]:
### Build a simple RAG chain with the hybrid retriever
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    RunnableMap(
        {
            "context": lambda x: format_docs(hybrid_retriever.invoke(x["question"])),
            "question": lambda x: x["question"],
        }
    )
    | prompt
    | llm
    | StrOutputParser()
)
rag_chain


{
  context: RunnableLambda(...),
  question: RunnableLambda(...)
}
| PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\nAnswer the question based on the context below.\n\nContext:\n{context}\n\nQuestion: {question}\n')
| ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x14d97d610>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x14dac7250>, model_name='llama-3.1-8b-instant', temperature=0.2, model_kwargs={}, groq_api_key=SecretStr('**********'))
| StrOutputParser()

In [9]:
# Step 9: Ask a question
query = {"question": "How can I build an app using LLMs?"}
answer = rag_chain.invoke(query)
source_docs = hybrid_retriever.invoke(query["question"])

print("Answer:\n", answer)
print("\nSource Documents:")
for i, doc in enumerate(source_docs, 1):
    print(f"\nDoc {i}: {doc.page_content}")


Answer:
 Based on the context, you can build an app using LLMs (Large Language Models) with the help of LangChain. LangChain is a platform that enables developers to build LLM applications, making it a suitable tool for creating such apps.

Source Documents:

Doc 1: LangChain helps build LLM applications.

Doc 2: LangChain can be used to develop agentic AI applications.

Doc 3: Pinecone is a vector database for semantic search.

Doc 4: LangChain has many types of retrievers.
